# LB 0.711 | YOLO Person Crop + R2Plus1D <100MB

**Verified public LB: 0.71144 (143/201), submission ref 55124182.**

This is the complete inference path for K-KUNO E290, published so another
participant can fork it and keep going. It uses two cross-subject R(2+1)D-34
folds, a fixed YOLO11n person crop, Depth_Color + IR, horizontal-flip TTA, and
signed int5/int6 storage. Classifier weights plus YOLO occupy **93,688,142
bytes**, below the Small Track's 100 MB cap.

The notebook never reads test labels, never displays test clips, and writes no
competition data to its output. Its only prediction artifact is
`submission.csv`.


## The short research story

- **Subject-CV, not random clips.** The test subjects are unseen; random splits
  overestimate progress through identity and room shortcuts.
- **One crop per clip.** YOLO probes eight IR frames, then one enlarged square
  union window is held fixed across all 16 model frames. A moving crop would
  cancel part of the action motion.
- **Strong video pretraining.** R(2+1)D-34 initialized through IG-65M and
  Kinetics-400 was difficult to beat with only about 3k training clips.
- **Diversity under the cap.** Two independently trained subject folds survive
  int5/int6 packing well enough to ensemble below 100 MB.
- **No leaderboard hand-labeling.** The public board is used only as a coarse
  calibration measurement.

See the companion `RESEARCH_HANDOFF.md` for negative results and next shots.


## License boundary

No CUHK-X frames, archives, labels, or caches are included in the model
dataset. Join the competition and accept its terms to run this notebook.

- Fine-tuned checkpoint: CUHK-X competition / permitted non-commercial
  research use, subject to CUHK-X License v2.0.
- YOLO11n: Ultralytics AGPL-3.0.
- IG65M architecture source: MIT, pinned commit
  `fc749e2ee354c3e4ddbb144cf511bb868b008f61`.
- Original notebook code: Apache-2.0.

Credit to CUHK AIoT Lab, the CUHK-X authors, Ultralytics, MoabitCoin's IG65M
implementation, and Kaggle user `welshonionman` for the original public YOLO
person-crop baseline.


In [1]:
from __future__ import annotations

import importlib.util
import io
import json
import math
import multiprocessing as mp
import os
import re
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
PIL_OPEN = Image.open  # keep the PNG reader before Ultralytics patches PIL
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Enable a Kaggle GPU for this notebook"
assert torch.cuda.get_device_capability(0) >= (7, 0), (
    "This Kaggle PyTorch image no longer ships sm_60 kernels; select a T4/L4 "
    "instead of P100 in notebook settings."
)
torch.set_float32_matmul_precision("high")
os.environ.setdefault("YOLO_AUTOINSTALL", "false")

N_CLASSES = 40
N_FRAMES = 16
DETECTION_FRAMES = 8
CHANNELS = 4
IMAGE_SIZE = 128
PERSON_CONFIDENCE = 0.25
CROP_MARGIN = 1.4
MIN_SIDE_FRACTION = 0.35
MICRO_BATCH = 8
NUM_WORKERS = 2

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
CACHE_ROOT = Path("/kaggle/temp/lb0711")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

PACKED_PATH = next(INPUT_ROOT.glob("**/ensemble_packed.pt"))
ASSET_ROOT = PACKED_PATH.parent
YOLO_PATH = ASSET_ROOT / "yolo11n.pt"
IG65M_SOURCE = ASSET_ROOT / "ig65m_models.py"

# The release verification uses a private, license-preserving cache source.
# Forks without that source fall back to the official gated Hugging Face ZIP.
# Accept the dataset terms, then add HF_TOKEN as a Kaggle secret; never paste a
# token into notebook source.
prebuilt_candidates = [
    path.parent for path in INPUT_ROOT.glob("**/test_frames.npy")
    if path.stat().st_size == 405 * N_FRAMES * CHANNELS * IMAGE_SIZE * IMAGE_SIZE
    and (path.parent / "test_meta.csv").is_file()
]
PREBUILT_CACHE = prebuilt_candidates[0] if prebuilt_candidates else None
TEST_ARCHIVE = None
TEST_CSV = None
if PREBUILT_CACHE is None:
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as error:
        raise RuntimeError(
            "Accept Kevin-Pal/CUHK-X_Small_Model_Track on Hugging Face and "
            "add HF_TOKEN as a Kaggle secret before forking this notebook."
        ) from error
    HF_REPO = "Kevin-Pal/CUHK-X_Small_Model_Track"
    HF_TEST_ARCHIVE = "Small-Model-Track/Testing/data/small_model_track_test.zip"
    HF_TEST_CSV = "Small-Model-Track/Testing/test_file/test.csv"
    SOURCE_ROOT = CACHE_ROOT / "official-source"
    TEST_ARCHIVE = Path(hf_hub_download(
        repo_id=HF_REPO, repo_type="dataset", filename=HF_TEST_ARCHIVE,
        local_dir=SOURCE_ROOT, token=hf_token,
    ))
    competition_csvs = [
        path for path in INPUT_ROOT.glob("**/test.csv")
        if "cuhk-x-competition-small-model-track" in str(path).casefold()
    ]
    TEST_CSV = competition_csvs[0] if competition_csvs else Path(hf_hub_download(
        repo_id=HF_REPO, repo_type="dataset", filename=HF_TEST_CSV,
        local_dir=SOURCE_ROOT, token=hf_token,
    ))
    del hf_token

assert YOLO_PATH.is_file() and IG65M_SOURCE.is_file()
if PREBUILT_CACHE is None:
    assert TEST_ARCHIVE.is_file() and TEST_CSV.is_file()
asset_bytes = PACKED_PATH.stat().st_size + YOLO_PATH.stat().st_size
assert asset_bytes == 93_688_142 and asset_bytes < 100_000_000
print({
    "torch": torch.__version__, "device": str(DEVICE),
    "input_mode": "private verified cache" if PREBUILT_CACHE else "official gated archive",
    "model_asset_bytes": asset_bytes,
})


{'torch': '2.10.0+cu128', 'device': 'cuda', 'input_mode': 'private verified cache', 'model_asset_bytes': 93688142}


## 1. Index the 405 test clips in submission order

The verified release run reads a private cache that is never exposed. A fork
without it fetches the sensor ZIP directly from the organizer-linked gated
Hugging Face dataset using the forker's Kaggle `HF_TOKEN` secret. It remains in
`/kaggle/temp` and is never emitted as notebook output. Only `Depth_Color` and
`IR` are read. Numeric frame sorting prevents frame 10 from appearing between
frames 1 and 2. Missing or unreadable frames become zeros.


In [2]:
NUMBER_PATTERN = re.compile(r"(\d+)")


@dataclass
class Clip:
    clip_id: str
    test_path: str
    depth: list[str]
    ir: list[str]


def natural_key(value: str):
    return tuple(
        (0, int(part)) if part.isdigit() else (1, part)
        for part in NUMBER_PATTERN.split(value.casefold()) if part
    )


test_clips = []
if PREBUILT_CACHE is not None:
    cached_meta = pd.read_csv(PREBUILT_CACHE / "test_meta.csv")
    test_table = cached_meta[["path"]].copy()
else:
    test_table = pd.read_csv(TEST_CSV)
    assert "path" in test_table and len(test_table) == 405
    members = {}
    member_pattern = re.compile(
        r"(?:^|/)small_model_track_test/(SM_test_\d{4})/(Depth_Color|IR)/([^/]+\.png)$"
    )
    with zipfile.ZipFile(TEST_ARCHIVE) as archive:
        seen_members = set()
        for info in archive.infolist():
            if info.filename in seen_members:
                raise RuntimeError(f"duplicate ZIP member: {info.filename}")
            seen_members.add(info.filename)
            if info.is_dir():
                continue
            match = member_pattern.search(info.filename)
            if match is None:
                continue
            clip_id, modality, filename = match.groups()
            members.setdefault(clip_id, {"Depth_Color": [], "IR": []})[modality].append(
                info.filename
            )
    for test_path in test_table["path"].astype(str):
        clip_id = test_path.strip("/").rsplit("/", 1)[-1]
        clip_members = members.get(clip_id, {"Depth_Color": [], "IR": []})
        test_clips.append(Clip(
            clip_id=clip_id, test_path=test_path,
            depth=sorted(clip_members["Depth_Color"], key=natural_key),
            ir=sorted(clip_members["IR"], key=natural_key),
        ))
    assert len({clip.clip_id for clip in test_clips}) == 405
    assert all(clip.depth or clip.ir for clip in test_clips)
print(f"ordered rows {len(test_table)} | raw clips indexed {len(test_clips)}")


ordered rows 405 | raw clips indexed 0


## 2. YOLO11n → one fixed crop per clip

Eight endpoint-uniform IR probes are processed in CPU batches, matching the
frozen E290 cache builder. For each clip we keep the highest-confidence person
box from each readable probe, union the boxes, expand by 1.4x, and make the
window square in the original 640x480 pixel geometry. No detection means a
full-frame fallback.


In [3]:
def pick_indices(length: int, count: int) -> tuple[int, ...]:
    if length <= 0:
        return ()
    return tuple(np.linspace(0, length - 1, count).round().astype(int).tolist())


def read_image(archive: zipfile.ZipFile, member: str, mode: str) -> Image.Image | None:
    try:
        payload = archive.read(member)
        if len(payload) > 16 * 1024 * 1024:
            return None
        with PIL_OPEN(io.BytesIO(payload)) as image:
            if image.format != "PNG":
                return None
            return image.convert(mode)
    except (KeyError, OSError, UnidentifiedImageError, ValueError, zipfile.BadZipFile):
        return None


def window_from_boxes(boxes):
    values = np.asarray(boxes, dtype=np.float64)
    x0, y0 = values[:, :2].min(axis=0)
    x1, y1 = values[:, 2:].max(axis=0)
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    width, height = 640.0, 480.0
    side = max((x1 - x0) * width, (y1 - y0) * height) * CROP_MARGIN
    side = max(side, MIN_SIDE_FRACTION * max(width, height))
    hx, hy = side / width / 2.0, side / height / 2.0
    return (
        max(cx - hx, 0.0), max(cy - hy, 0.0),
        min(cx + hx, 1.0), min(cy + hy, 1.0),
    )


def detect_windows(archive: zipfile.ZipFile, clips: list[Clip], batch_size: int = 32):
    from ultralytics import YOLO

    model = YOLO(str(YOLO_PATH))
    boxes_by_clip = [[] for _ in clips]
    frame_batch, owner_batch = [], []
    readable = 0

    def flush():
        nonlocal readable
        if not frame_batch:
            return
        results = model.predict(
            frame_batch, classes=[0], conf=PERSON_CONFIDENCE,
            verbose=False, device="cpu", batch=batch_size,
        )
        assert len(results) == len(frame_batch)
        for owner, result in zip(owner_batch, results, strict=True):
            readable += 1
            if len(result.boxes):
                box = result.boxes.xyxy[result.boxes.conf.argmax()].tolist()
                height, width = result.orig_shape
                boxes_by_clip[owner].append((
                    float(box[0]) / width, float(box[1]) / height,
                    float(box[2]) / width, float(box[3]) / height,
                ))
        frame_batch.clear()
        owner_batch.clear()

    for owner, clip in enumerate(tqdm(clips, desc="queue YOLO probes", unit="clip")):
        for index in sorted(set(pick_indices(len(clip.ir), DETECTION_FRAMES))):
            image = read_image(archive, clip.ir[index], "RGB")
            if image is None:
                continue
            array = np.asarray(image)
            if not array.any():
                continue
            frame_batch.append(array)
            owner_batch.append(owner)
            if len(frame_batch) >= batch_size:
                flush()
    flush()
    windows = [window_from_boxes(boxes) if boxes else None for boxes in boxes_by_clip]
    del model
    torch.cuda.empty_cache()
    print({
        "readable_probe_frames": readable,
        "detected_clips": sum(window is not None for window in windows),
        "fallback_clips": sum(window is None for window in windows),
    })
    return windows


started = time.time()
test_windows = []
if PREBUILT_CACHE is None:
    with zipfile.ZipFile(TEST_ARCHIVE) as test_archive:
        test_windows = detect_windows(test_archive, test_clips)
print(f"person detection: {(time.time() - started) / 60:.1f} min")


person detection: 0.0 min


## 3. Decode 16 Depth_Color + IR frames

The cache is a raw uint8 memmap in `/kaggle/temp`, so it is not published with
the notebook output. Endpoint-uniform sampling and bilinear resize match E290.
All-zero frames remain zero and are counted only as aggregate diagnostics.


In [4]:
TEST_CACHE = (
    PREBUILT_CACHE / "test_frames.npy"
    if PREBUILT_CACHE is not None else CACHE_ROOT / "test_frames.npy"
)
TEST_META = (
    PREBUILT_CACHE / "test_meta.csv"
    if PREBUILT_CACHE is not None else CACHE_ROOT / "test_meta.csv"
)
TEST_SHAPE = (405, N_FRAMES, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
MM = None
WORKER_ARCHIVE = None


def init_worker(path, shape, archive_path):
    global MM, WORKER_ARCHIVE
    MM = np.memmap(path, dtype=np.uint8, mode="r+", shape=shape)
    WORKER_ARCHIVE = zipfile.ZipFile(archive_path)


def process_clip(job):
    row, clip, window = job
    output = np.zeros(MM.shape[1:], dtype=np.uint8)
    bad_depth = bad_ir = 0
    for members, mode, begin, width in (
        (clip.depth, "RGB", 0, 3),
        (clip.ir, "L", 3, 1),
    ):
        if not members:
            if mode == "RGB": bad_depth = N_FRAMES
            else: bad_ir = N_FRAMES
            continue
        for frame, index in enumerate(pick_indices(len(members), N_FRAMES)):
            image = read_image(WORKER_ARCHIVE, members[index], mode)
            if image is None:
                if mode == "RGB": bad_depth += 1
                else: bad_ir += 1
                continue
            if window is not None:
                image_width, image_height = image.size
                image = image.crop((
                    round(window[0] * image_width), round(window[1] * image_height),
                    round(window[2] * image_width), round(window[3] * image_height),
                ))
            array = np.asarray(
                image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR),
                dtype=np.uint8,
            )
            if not array.any():
                if mode == "RGB": bad_depth += 1
                else: bad_ir += 1
                continue
            output[frame, begin:begin + width] = (
                array.transpose(2, 0, 1) if width == 3 else array[None]
            )
    MM[row] = output
    return {
        "row": row, "clip_id": clip.clip_id, "path": clip.test_path,
        "bad_depth_frames": bad_depth, "bad_ir_frames": bad_ir,
        "has_crop": window is not None,
    }


if not (TEST_CACHE.is_file() and TEST_CACHE.stat().st_size == int(np.prod(TEST_SHAPE)) and TEST_META.is_file()):
    np.memmap(TEST_CACHE, dtype=np.uint8, mode="w+", shape=TEST_SHAPE).flush()
    context = mp.get_context("fork")
    rows = []
    with context.Pool(
        4, initializer=init_worker,
        initargs=(str(TEST_CACHE), TEST_SHAPE, str(TEST_ARCHIVE)),
    ) as pool:
        jobs = zip(range(len(test_clips)), test_clips, test_windows)
        for record in tqdm(
            pool.imap_unordered(process_clip, jobs, chunksize=8),
            total=len(test_clips), desc="decode test", unit="clip",
        ):
            rows.append(record)
    test_meta = pd.DataFrame(rows).sort_values("row").reset_index(drop=True)
    test_meta.to_csv(TEST_META, index=False)
else:
    test_meta = pd.read_csv(TEST_META)

assert test_meta["row"].tolist() == list(range(405))
assert test_meta["path"].tolist() == test_table["path"].astype(str).tolist()
print({
    "cache_shape": TEST_SHAPE,
    "bad_depth_frames": int(test_meta["bad_depth_frames"].sum()),
    "bad_ir_frames": int(test_meta["bad_ir_frames"].sum()),
    "crop_fallbacks": int((~test_meta["has_crop"].astype(bool)).sum()),
})


{'cache_shape': (405, 16, 4, 128, 128), 'bad_depth_frames': 0, 'bad_ir_frames': 64, 'crop_fallbacks': 10}


## 4. Build R(2+1)D-34 and unpack the two classifiers

The signed low-bit representation is storage quantization: weights are packed
below int8 on disk and dequantized to floating point for inference. Scales are
per output channel. The exact IG65M architecture source is attached with the
model, avoiding a moving `torch.hub` dependency.


In [5]:
spec = importlib.util.spec_from_file_location("ig65m_models", IG65M_SOURCE)
ig65m_models = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(ig65m_models)

KINETICS_MEAN = (0.43216, 0.394666, 0.37645)
KINETICS_STD = (0.22803, 0.22145, 0.216989)


class TestDataset(Dataset):
    def __init__(self, cache_path: Path, metadata: pd.DataFrame):
        self.path = str(cache_path)
        self.metadata = metadata.reset_index(drop=True)
        self.rows = self.metadata["row"].to_numpy(dtype=np.int64)
        mean = (*KINETICS_MEAN, sum(KINETICS_MEAN) / 3.0)
        std = (*KINETICS_STD, sum(KINETICS_STD) / 3.0)
        self.mean = torch.tensor(mean).view(1, CHANNELS, 1, 1)
        self.std = torch.tensor(std).view(1, CHANNELS, 1, 1)
        self.memmap = None

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        if self.memmap is None:
            self.memmap = np.memmap(
                self.path, dtype=np.uint8, mode="r", shape=TEST_SHAPE
            )
        clip = torch.from_numpy(
            np.asarray(self.memmap[self.rows[index]]).copy()
        ).float().div_(255.0)
        return clip.sub_(self.mean).div_(self.std), index


def adapt_input_conv(conv: nn.Conv3d, channels: int) -> nn.Conv3d:
    replacement = type(conv)(
        channels, conv.out_channels, conv.kernel_size,
        conv.stride, conv.padding, bias=conv.bias is not None,
    )
    with torch.no_grad():
        replacement.weight[:, :3] = conv.weight
        replacement.weight[:, 3:] = conv.weight.mean(
            dim=1, keepdim=True
        ).expand(-1, channels - 3, -1, -1, -1)
        if conv.bias is not None:
            replacement.bias.copy_(conv.bias)
    return replacement


class R2Plus1D34(nn.Module):
    def __init__(self):
        super().__init__()
        network = ig65m_models.r2plus1d_34_32_kinetics(
            num_classes=400, pretrained=False
        )
        network.stem[0] = adapt_input_conv(network.stem[0], CHANNELS)
        features = network.fc.in_features
        network.fc = nn.Identity()
        self.encoder = network
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(features, N_CLASSES))

    def forward(self, inputs):
        return self.head(self.encoder(inputs.permute(0, 2, 1, 3, 4)))


def unpack_signed(packed: torch.Tensor, shape: tuple[int, ...], bits: int):
    count = math.prod(shape)
    starts = torch.arange(count, dtype=torch.int64) * bits
    codes = torch.zeros(count, dtype=torch.int16)
    source = packed.to(torch.int16)
    for bit in range(bits):
        positions = starts + bit
        byte_indices = positions >> 3
        shifts = positions & 7
        values = torch.bitwise_and(source[byte_indices] >> shifts, 1)
        codes |= values << bit
    sign, modulus = 1 << (bits - 1), 1 << bits
    signed = torch.where(codes >= sign, codes - modulus, codes)
    return signed.to(torch.int8).reshape(shape)


def dequantize_state(state: Mapping[str, object]):
    output = {}
    for key, value in state.items():
        if isinstance(value, Mapping):
            shape = tuple(int(item) for item in value["shape"])
            quantized = unpack_signed(value["packed"], shape, int(value["bits"]))
            output[key] = quantized.float() * value["scale"].float()
        else:
            output[key] = value.float() if value.is_floating_point() else value
    return output


checkpoint = torch.load(PACKED_PATH, map_location="cpu", weights_only=True)
assert checkpoint["schema_version"] == "kuno-yolo-r2p1d-packed-ensemble/v1"
assert checkpoint["bits"] == [5, 6]
assert checkpoint["folds"] == [0, 1]
assert checkpoint["weights"] == [0.5, 0.5]
assert len(checkpoint["models_packed"]) == 2
print({
    "packed_bits": checkpoint["bits"],
    "folds": checkpoint["folds"],
    "parent_subject_val_accuracy": checkpoint["parent_val_acc"],
})


{'packed_bits': [5, 6], 'folds': [0, 1], 'parent_subject_val_accuracy': [0.7155963302752294, 0.7187039764359352]}


## 5. Two-fold logit ensemble + horizontal-flip TTA

Each model sees the original clip and its horizontal mirror. We sum logits,
then average the two folds with equal weight. Models run sequentially to keep
GPU memory modest.


In [6]:
test_loader = DataLoader(
    TestDataset(TEST_CACHE, test_meta),
    batch_size=MICRO_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)


@torch.inference_mode()
def predict_logits(packed_state, description):
    model = R2Plus1D34().to(DEVICE)
    state = dequantize_state(packed_state)
    model.load_state_dict(state)
    del state
    model.eval()
    logits = np.zeros((len(test_loader.dataset), N_CLASSES), dtype=np.float32)
    for inputs, indices in tqdm(test_loader, desc=description, unit="batch"):
        inputs = inputs.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            output = model(inputs) + model(torch.flip(inputs, dims=(-1,)))
        logits[indices.numpy()] = output.float().cpu().numpy()
    del model
    torch.cuda.empty_cache()
    return logits


started = time.time()
ensemble_logits = np.zeros((405, N_CLASSES), dtype=np.float32)
for weight, fold, bits, packed_state in zip(
    checkpoint["weights"], checkpoint["folds"],
    checkpoint["bits"], checkpoint["models_packed"], strict=True,
):
    ensemble_logits += np.float32(weight) * predict_logits(
        packed_state, f"fold {fold} / int{bits}"
    )

predictions = ensemble_logits.argmax(axis=1).astype(np.int64)
submission = pd.DataFrame({
    "path": test_meta["path"].astype(str),
    "prediction": predictions,
})
assert list(submission.columns) == ["path", "prediction"]
assert len(submission) == 405
assert submission["path"].tolist() == test_table["path"].astype(str).tolist()
assert submission["prediction"].between(0, N_CLASSES - 1).all()
assert submission["prediction"].nunique() >= 35

submission.to_csv(WORK_ROOT / "submission.csv", index=False)
report = {
    "status": "PASS",
    "candidate": "ksv1-e290-yolo-r2p1d34-int5-int6-ensemble-v1",
    "rows": len(submission),
    "prediction_classes": int(submission["prediction"].nunique()),
    "model_asset_bytes": asset_bytes,
    "input_mode": "private verified cache" if PREBUILT_CACHE else "official gated archive",
    "elapsed_minutes": (time.time() - started) / 60.0,
    "test_labels_opened": False,
    "test_frames_displayed": False,
}
(WORK_ROOT / "run_report.json").write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(report)
submission.head()


fold 0 / int5:   0%|          | 0/51 [00:00<?, ?batch/s]

fold 1 / int6:   0%|          | 0/51 [00:00<?, ?batch/s]

{'status': 'PASS', 'candidate': 'ksv1-e290-yolo-r2p1d34-int5-int6-ensemble-v1', 'rows': 405, 'prediction_classes': 39, 'model_asset_bytes': 93688142, 'input_mode': 'private verified cache', 'elapsed_minutes': 0.9851651748021444, 'test_labels_opened': False, 'test_frames_displayed': False}


,path,prediction
0,small_model_track_test/SM_test_0001/,35
1,small_model_track_test/SM_test_0002/,13
2,small_model_track_test/SM_test_0003/,11
3,small_model_track_test/SM_test_0004/,29
4,small_model_track_test/SM_test_0005/,2


## Continue the work

The strongest next direction is a **genuinely complementary signal**, not a
large blend sweep around this prediction vector. Good candidates are a compact
skeleton/radar branch, crop-robust training, or a different video family chosen
for subject-OOF error complementarity. Keep fixed subject folds, keep the final
inference package under 100 MB, and use Kaggle only for honest calibration.

If you improve it, please publish the method and give the original dataset and
component authors their attribution. Good luck — the ceiling is not sacred.
